# Building a RAG application from scratch using LangChain & Pinecone


### Loading the environment variables

In [1]:
import getpass
import os

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")   

OPENAI_API_KEY: ········


In [2]:
_set_env("PINECONE_API_KEY")

PINECONE_API_KEY: ········


### Setting up the model 

In [5]:
!pip install langchain_openai

from langchain_openai.chat_models import ChatOpenAI

# Retrieve the API key directly from the environment variables when creating the model instance
model = ChatOpenAI(openai_api_key=os.getenv("OPENAI_API_KEY"), model="gpt-3.5-turbo")


In [6]:
model

ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x000002688AA4CE90>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x000002688AA54C10>, openai_api_key=SecretStr('**********'), openai_proxy='')

### Test the model by asking a simple question

In [7]:
model.invoke("Name 3 yellow spring flowers.")

AIMessage(content='1. Daffodil\n2. Forsythia\n3. Tulip', response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 14, 'total_tokens': 31}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-da7f5e56-088c-439d-b8a0-274f059a29d6-0', usage_metadata={'input_tokens': 14, 'output_tokens': 17, 'total_tokens': 31})

Extract this answer by chaining the model with an output parser. So we use a simple StrOutputParser to extract the answer as a string.

In [8]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = model | parser
chain.invoke("Name 3 yellow spring flowers.")


'1. Daffodil\n2. Forsythia\n3. Tulip'

### Introducing prompt templates

We want to provide the model with some context and the question. Prompt templates are a simple way to define and reuse prompts.

In [9]:
from langchain.prompts import ChatPromptTemplate

template = """
Answer the question based on the context below. If you can't
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt.format(context="Varsha lives in Bangalore", question="Where does Varsha live?")


'Human: \nAnswer the question based on the context below. If you can\'t\nanswer the question, reply "I don\'t know".\n\nContext: Varsha lives in Bangalore\n\nQuestion: Where does Varsha live?\n'

chain the prompt with the model and the output parser

In [10]:
chain = prompt | model | parser
chain.invoke({
    "context": "Varsha lives in Bangalore",
    "question": "Where does Varsha live?"
})

'Varsha lives in Bangalore.'

### Combining chains

We can combine different chains to create more complex workflows. creating a new prompt template for the translation chain

In [11]:
translation_prompt = ChatPromptTemplate.from_template(
    "Translate {answer} to {language}"
)

creating a new translation chain that combines the result from the first chain with the translation prompt.

In [12]:
from operator import itemgetter

translation_chain = (
    {"answer": chain, "language": itemgetter("language")} | translation_prompt | model | parser
)

translation_chain.invoke(
    {
        "context": "Varsha likes Ice-cream. Varsha's brother name is Badal.",
        "question": "How many siblings does Varsha have?",
        "language": "Spanish",
    }
)

'Varsha tiene un hermano, Badal.'

### Loading the transcription file

Read the transcription and display the first few characters to ensure everything works as expected

In [13]:
with open("transcription.txt") as file:
    transcription = file.read()

transcription[:102]


"- I think it's possible that physics has exploits and we should be trying to find them. Arranging some"

### Using the entire transcription as context

If we try to invoke the chain using the transcription as context, the model will return an error because the context is too long. Large Language Models support limitted context sizes. so we need to find a different solution (Splitting).

In [14]:
try:
    chain.invoke({
        "context": transcription,
        "question": "Is reading papers a good idea?"
    })
except Exception as e:
    print(e)

Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 49091 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}


### Splitting the transcription

Since we can’t use the entire transcription as the context for the model, a potential solution is to split the transcription into smaller chunks. 
We can then invoke the model using only the relevant chunks to answer a particular question.
Let’s start by loading the transcription in memory

In [15]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("transcription.txt")
text_documents = loader.load()

In [16]:
type(text_documents)

list

There are many different ways to split a document. For this example, we’ll use a simple splitter that splits the document into chunks of a fixed size.
split the transcription into chunks of 1000 characters with an overlap of 20 characters.

In [17]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
documents = text_splitter.split_documents(text_documents)

In [18]:
type(documents[0])

langchain_core.documents.base.Document

In [19]:
documents[15]

Document(metadata={'source': 'transcription.txt'}, page_content='But that seems crazy \'cause how many single-cell organisms are there?\nAnd how much time you have surely it\'s not that difficult. And a billion years is not even that long\nof a time really, just all these bacteria under constrained resources battling it out.\nI\'m sure they can invent more complex. I don\'t understand. It\'s like how to move from a "Hello, World!" program\nto invent a function or something like that. I don\'t- - Yeah.\n- Yeah, so I\'m with you. I just feel like I don\'t see any, if the origin of life, that would be my intuition, that\'s the hardest thing.\nBut if that\'s not the hardest thing \'cause it happened so quickly, then it\'s gotta be everywhere and yeah, maybe we\'re just too dumb to see it.\n- Well, it\'s just we don\'t have really good mechanisms for seeing this life.\nSo I\'m not an expert just to preface this but just from what- - On aliens? I wanna meet an expert on alien intelligence')

In [20]:
len(documents)

230

### Constructing the model with openai embeddings

we need to find the relevant chunks from the transcription to send to the model. Here is where the idea of embeddings comes into play.
To provide with the most relevant chunks, we can use the embeddings of the question and the chunks of the transcription to compute the similarity between them. We can then select the chunks with the highest similarity to the question and use them as the context for the model


Generate Embeddings

In [21]:
from langchain_openai.embeddings import OpenAIEmbeddings
model_name = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(
    model = model_name,
    openai_api_key = os.environ.get("OPENAI_API_KEY")
)

### Setting up a Vector Store

Constructing the vector store with pinecone

We need an efficient way to store document chunks, their embeddings, and perform similarity searches at scale. To do this, we’ll use a vector store.
A vector store is a database of embeddings that specializes in fast similarity searches.

Connecting the vector store to the chain
We can use the vector store to find the most relevant chunks from the transcription to send to the model. Here is how we can connect the vector store to the chain

#### Setting up Pinecone

The first step is to create a Pinecone account, set up an index, get an API key, and set it as an environment variable PINECONE_API_KEY.

In [22]:
!pip install langchain_pinecone --user

Load the transcription documents into Pinecone by constructing the index and adding name to it.

In [24]:
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
# Creates an index using the API key stored in the client 'pc'.
pc.create_index(
    name= "ram-rag-index",
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(
        cloud= 'aws',
        region= 'us-east-1'
    )
)

connecting to index

In [25]:
# connect to index
index = pc.Index("ram-rag-index")
# view index stats
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

In [26]:
from langchain_pinecone import PineconeVectorStore
pinecone =PineconeVectorStore.from_documents(
    documents, embeddings, index_name= 'ram-rag-index'
)

In [27]:
pinecone

### Convert the vector store into a retriever

Run a similarity search on pinecone to make sure everything works

In [28]:
pinecone.similarity_search("what is tesla's data engine?")[:3]

[Document(metadata={'source': 'transcription.txt'}, page_content="that the team is doing to make sure it all fits and utilizes the engine. So I think it's extremely good engineering\nand then there's all kinds of little insights peppered in on how to do it properly. - Let's actually zoom out\nTesla's Data Engine\n'cause I don't think we talked about the data engine, the entirety of the layouts of this idea\nthat I think is just beautiful with humans in the loop. Can you describe the data engine?\n- Yeah, the data engine is what I call the almost biological feeling process\nby which you perfect the training sets for these neural networks.\nSo because most of the programming now is in the level of these data sets and make sure they're large, diverse and clean, basically you have a data set\nthat you think is good, you train your neural net, you deploy it, and then you observe how well it's performing\nand you're trying to always increase the quality of your data set. So you're trying to 

setup the new chain using Pinecone as the vector store

In [29]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": pinecone.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | model
    | parser
)
chain.invoke("What is Tesla's Data Engine?")

'The Tesla Data Engine is described as a process by which training sets for neural networks are perfected, with a focus on ensuring the data sets are large, diverse, and clean. The process involves training neural networks with the data sets, deploying them, observing their performance, and constantly improving the quality of the data sets.'